In [41]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [42]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [50]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 16), # 16 layer, each layer has 4 weights -> output 16 values
            nn.ReLU(),
            nn.Linear(16, 8), # 8 layer, each Layer has 16 weights -> output 8 values
            nn.ReLU(),
            nn.Linear(8, 3) # 3 layer, each layer has 8 weights -> output 3 values
            # nn.Softmax(dim = 1)
        )
        # total weights = (14 * 4 + 16) + (17 * 8 + 8) + (8 * 3 + 3)

    def forward(self, x):
        return self.model(x)

model = MLP()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epoch}, Loss: {loss.item()}")
    if loss < 1e-2:
        break

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    y_pred_classes = torch.argmax(y_pred, dim=1)
    accuracy = accuracy_score(y_test, y_pred_classes)

print(f"ACcuracy on test set: {accuracy:.4f}")

Epoch 10/9, Loss: 0.7752543091773987
Epoch 20/19, Loss: 0.5697900056838989
Epoch 30/29, Loss: 0.4285150170326233
Epoch 40/39, Loss: 0.286056786775589
Epoch 50/49, Loss: 0.14765401184558868
Epoch 60/59, Loss: 0.07598408311605453
Epoch 70/69, Loss: 0.05427052080631256
Epoch 80/79, Loss: 0.046884216368198395
Epoch 90/89, Loss: 0.04378855973482132
Epoch 100/99, Loss: 0.04149148240685463
ACcuracy on test set: 0.9667
